# Duration-based runs: backlog progress UI

When LLMeter sends requests at maximum concurrency in a duration-bound run, the asyncio event loop can become saturated. This means the result-processing queue builds up a backlog that takes time to drain after clients stop sending.

Previously, users saw no feedback during this draining phase — it looked like the test was stuck. Now LLMeter shows a clear two-phase UI:

1. **Sending phase** — a time-based progress bar counts elapsed seconds while clients send requests
2. **Processing backlog** — once sending stops, a second progress bar shows remaining results being processed

The live stats table continues updating throughout both phases.

This notebook demonstrates the behavior using Amazon Nova Micro on Bedrock — a fast, cheap model that makes it easy to generate enough load for a visible backlog.

In [ ]:
%pip install "llmeter[plotting]<1" boto3

In [ ]:
from llmeter.endpoints import BedrockConverseStream
from llmeter.runner import Runner

## Setup

We use **Amazon Nova Micro** (`amazon.nova-micro-v1:0`) — a text-only model that's fast and inexpensive.
Make sure you have [model access enabled](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html) in your region.

We configure it to produce short responses (`max_tokens=100`) so each request completes quickly, allowing us to saturate the event loop with many concurrent clients.

In [ ]:
MODEL_ID = "amazon.nova-micro-v1:0"
AWS_REGION = "us-east-1"  # Change to your region

endpoint = BedrockConverseStream(
    model_id=MODEL_ID,
    region=AWS_REGION,
)

In [ ]:
# Quick connectivity check
payload = BedrockConverseStream.create_payload("Say hello in one sentence.", max_tokens=50)
response = endpoint.invoke(payload)
print(response)

## Duration-based run with high concurrency

This is where the backlog UI becomes visible. We run for a short duration (30 seconds) but with many concurrent clients (50). Because each client fires requests as fast as it can, the result-processing queue accumulates a backlog.

Watch for:
1. The **"Elapsed"** progress bar counting up to 30s
2. After 30s, the elapsed bar closes and a **"Processing backlog"** bar appears showing remaining results
3. The stats table continues updating with `reqs=N` throughout both phases

In [ ]:
runner = Runner(
    endpoint=endpoint,
    output_path=f"outputs/{MODEL_ID}-duration-backlog-demo",
)

payload = BedrockConverseStream.create_payload(
    "Explain cloud computing in exactly 3 bullet points.",
    max_tokens=100,
)

result = await runner.run(
    payload=payload,
    clients=50,
    run_duration=30,  # seconds
    run_name="backlog-demo-50-clients-30s",
)

In [ ]:
print(f"Total requests completed: {result.total_requests}")
print(f"Total test time: {result.total_test_time:.1f}s")
print(f"Requests per minute: {result.stats.get('requests_per_minute', 0):.1f}")
print(f"Failed requests: {result.stats.get('failed_requests', 0)}")

In [ ]:
result.stats

## Comparison: low concurrency (no backlog)

With fewer clients, the event loop isn't saturated and there's no significant backlog. The elapsed bar completes and results finish processing almost immediately — no backlog bar appears.

In [ ]:
result_low = await runner.run(
    payload=payload,
    clients=3,
    run_duration=15,  # seconds
    run_name="no-backlog-3-clients-15s",
)

In [ ]:
print(f"Total requests completed: {result_low.total_requests}")
print(f"Total test time: {result_low.total_test_time:.1f}s")
print(f"Requests per minute: {result_low.stats.get('requests_per_minute', 0):.1f}")

## What's happening under the hood

In duration-bound mode, LLMeter now uses a **two-phase approach**:

**Phase 1: Sending** — All clients send requests concurrently for the specified duration. A time-based progress bar ticks every 0.5s. The result processor runs in the background, counting tokens and updating live stats.

**Phase 2: Draining** — Once the duration expires, clients stop sending. The time bar closes at 100%. If there are still unprocessed results in the queue, a "Processing backlog" bar appears showing the remaining count. Stats continue updating as each result is processed.

This gives clear visual feedback that:
- The test has **stopped sending** requests (time bar is done)
- Results are still being **processed** from the backlog (count-based bar)
- Stats are **accurate** and keep updating throughout